In [2]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
import csv
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
import sys
np.set_printoptions(threshold=sys.maxsize)

In [3]:
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

df_raw = pd.read_csv("Dataset_insuficiencia_renal_crnica.csv")

df_work = df_raw.copy(deep=True)


print("Raw dataset shape:", df_raw.shape)
print("Working dataset shape:", df_work.shape)

print("Exact duplicate rows:", df_work.duplicated().sum())
print("Missing target values:",
      df_work["Insuficiencia renal crónica"].isna().sum())

Raw dataset shape: (15051, 21)
Working dataset shape: (15051, 21)
Exact duplicate rows: 0
Missing target values: 2


In [4]:
df_work.info()

print("\n--- Column names ---")
print(df_work.columns.tolist())

print("\n--- Missing values ---")
print(
    df_work.isna()
    .sum()
    .sort_values(ascending=False)
)

<class 'pandas.DataFrame'>
RangeIndex: 15051 entries, 0 to 15050
Data columns (total 21 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Identificación del paciente      15051 non-null  int64  
 1   Edad                             15051 non-null  int64  
 2   Presión arterial                 15051 non-null  int64  
 3   Gravedad específica de la orina  15051 non-null  float64
 4   Albúmina en orina                15051 non-null  float64
 5   Azúcar en orina                  15051 non-null  int64  
 6   Glóbulos rojos                   15051 non-null  str    
 7   Células de pus                   15051 non-null  str    
 8   Grupos de células de pus         15042 non-null  str    
 9   Bacterias                        15045 non-null  str    
 10  Glucosa en sangre aleatoria      15051 non-null  int64  
 11  Urea en sangre                   15051 non-null  float64
 12  Creatinina sérica            

In [5]:
categorical_cols = [
    "Glóbulos rojos",
    "Células de pus",
    "Grupos de células de pus",
    "Bacterias",
    "Apetito",
    "Insuficiencia renal crónica"
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(df_work[col].value_counts(dropna=False))
    print("Unique values:", df_work[col].nunique(dropna=True))


--- Glóbulos rojos ---
Glóbulos rojos
normal     7690
anormal    7361
Name: count, dtype: int64
Unique values: 2

--- Células de pus ---
Células de pus
normal     7655
anormal    7396
Name: count, dtype: int64
Unique values: 2

--- Grupos de células de pus ---
Grupos de células de pus
ausente     12596
presente     2446
NaN             9
Name: count, dtype: int64
Unique values: 2

--- Bacterias ---
Bacterias
ausente     13732
presente     1313
NaN             6
Name: count, dtype: int64
Unique values: 2

--- Apetito ---
Apetito
bueno    8425
bajo     6625
NaN         1
Name: count, dtype: int64
Unique values: 2

--- Insuficiencia renal crónica ---
Insuficiencia renal crónica
no     7696
si     7353
NaN       2
Name: count, dtype: int64
Unique values: 2


In [6]:
numeric_cols = df_work.select_dtypes(include="number").columns

print(df_work[numeric_cols].describe().T)

                                   count    mean     std     min     25%  \
Identificación del paciente     15051.00 7525.00 4344.99    0.00 3762.50   
Edad                            15051.00   52.10   17.55    2.00   45.00   
Presión arterial                15051.00   93.23   19.91   50.00   80.00   
Gravedad específica de la orina 15051.00  541.34  502.98   10.00   12.63   
Albúmina en orina               15051.00  228.36  314.22    0.00    0.00   
Azúcar en orina                 15051.00  154.32  248.58    0.00    0.00   
Glucosa en sangre aleatoria     15051.00  157.61   62.35   70.00  105.00   
Urea en sangre                  15051.00   59.65   42.51   10.00   25.00   
Creatinina sérica               15042.00    3.22    5.48    0.40    0.93   
Sodio en sangre                 15051.00  138.58    5.82  120.00  135.00   
Potasio en sangre               15051.00    5.38    0.86    3.50    4.79   
Hemoglobina                     15051.00   12.00    2.41    6.00   10.00   
Volumen corp

In [8]:
review_masks = {
    "Specific Gravity < 1000":
        df_work["Gravedad específica de la orina"] < 1000,

    "Albumin > 5":
        df_work["Albúmina en orina"] > 5,

    "Sugar > 5":
        df_work["Azúcar en orina"] > 5,

    "Creatinine > 6.73":
        df_work["Creatinina sérica"] > 6.73
}

for label, mask in review_masks.items():
    print(f"{label}: {mask.sum()} records")

print("\n--- Records with unusual Specific Gravity, Albumin, Sugar ---")

cols_to_review = [
    "Identificación del paciente",
    "Gravedad específica de la orina",
    "Albúmina en orina",
    "Azúcar en orina",
    "Creatinina sérica",
    "Insuficiencia renal crónica"
]

print(
    df_work.loc[
        review_masks["Specific Gravity < 1000"]
        & review_masks["Albumin > 5"]
        & review_masks["Sugar > 5"],
        cols_to_review
    ].head(10)
)

Specific Gravity < 1000: 7148 records
Albumin > 5: 7077 records
Sugar > 5: 7148 records
Creatinine > 6.73: 1536 records

--- Records with unusual Specific Gravity, Albumin, Sugar ---
      Identificación del paciente  Gravedad específica de la orina  \
7892                         7892                            12.53   
7893                         7893                            11.36   
7894                         7894                            12.65   
7895                         7895                            11.78   
7896                         7896                            12.08   
7897                         7897                            12.14   
7898                         7898                            10.51   
7899                         7899                            14.88   
7900                         7900                            14.37   
7901                         7901                            13.81   

      Albúmina en orina  Azúcar en orina  Crea

In [9]:
df_work["SG_unusual"] = (
    df_work["Gravedad específica de la orina"] < 1000
)

df_work["Albumin_unusual"] = (
    df_work["Albúmina en orina"] > 5
)

df_work["Sugar_unusual"] = (
    df_work["Azúcar en orina"] > 5
)

df_work["ID_group"] = pd.cut(
    df_work["Identificación del paciente"],
    bins=[-1, 3000, 6000, 9000, 12000, 15050],
    labels=["0–3000", "3001–6000", "6001–9000",
            "9001–12000", "12001–15050"]
)

print(
    df_work.groupby("ID_group", observed=False)[
        ["SG_unusual", "Albumin_unusual", "Sugar_unusual"]
    ].mean().mul(100).round(2)
)

             SG_unusual  Albumin_unusual  Sugar_unusual
ID_group                                               
0–3000             0.00             0.00           0.00
3001–6000          0.00             0.00           0.00
6001–9000         36.97            36.97          36.97
9001–12000       100.00            97.63         100.00
12001–15050       99.64            99.64          99.64


In [10]:
target = "Insuficiencia renal crónica"
id_col = "Identificación del paciente"


missing_target_rows = df_work.loc[
    df_work[target].isna()
].copy()


df_work = df_work.dropna(subset=[target]).copy()

print("Rows excluded due to missing target:",
      len(missing_target_rows))

print("Working dataset shape:", df_work.shape)

print("Missing target values:",
      df_work[target].isna().sum())

print("Raw dataset shape (unchanged):", df_raw.shape)

Rows excluded due to missing target: 2
Working dataset shape: (15049, 25)
Missing target values: 0
Raw dataset shape (unchanged): (15051, 21)


In [11]:
missing_summary = (
    df_work.isna()
    .sum()
    .sort_values(ascending=False)
)

print(missing_summary[missing_summary > 0])


feature_cols = [
    col for col in df_work.columns
    if col not in [
        "Insuficiencia renal crónica",
        "SG_unusual",
        "Albumin_unusual",
        "Sugar_unusual",
        "ID_group"
    ]
]

print(
    "\nRows with at least one missing feature:",
    df_work[feature_cols].isna().any(axis=1).sum()
)

Grupos de células de pus    9
Creatinina sérica           9
Bacterias                   6
Apetito                     1
dtype: int64

Rows with at least one missing feature: 25


In [12]:
target = "Insuficiencia renal crónica"
id_col = "Identificación del paciente"

missing_summary = pd.DataFrame({
    "Missing Count": df_work.isna().sum(),
    "Missing %": (df_work.isna().mean() * 100).round(2),
    "Data Type": df_work.dtypes.astype(str)
})

missing_summary = missing_summary[
    missing_summary["Missing Count"] > 0
].sort_values("Missing Count", ascending=False)

display(missing_summary)

,Missing Count,Missing %,Data Type
Grupos de células de pus,9,0.06,str
Creatinina sérica,9,0.06,float64
Bacterias,6,0.04,str
Apetito,1,0.01,str


In [13]:
cols_to_inspect = [
    "Grupos de células de pus",
    "Creatinina sérica",
    "Bacterias",
    "Apetito"
]

for col in cols_to_inspect:
    print(f"\n--- {col} ---")
    print(df_work[col].value_counts(dropna=False).head(10))

    if pd.api.types.is_numeric_dtype(df_work[col]):
        print(df_work[col].describe())
        print("Median:", df_work[col].median())


--- Grupos de células de pus ---
Grupos de células de pus
ausente     12595
presente     2445
NaN             9
Name: count, dtype: int64

--- Creatinina sérica ---
Creatinina sérica
1.20    949
0.90    771
1.00    641
0.70    576
1.10    571
0.50    418
0.60    416
0.80    405
3.20    352
1.30    328
Name: count, dtype: int64
count   15040.00
mean        3.22
std         5.48
min         0.40
25%         0.93
50%         1.40
75%         3.25
max        76.00
Name: Creatinina sérica, dtype: float64
Median: 1.4

--- Bacterias ---
Bacterias
ausente     13731
presente     1312
NaN             6
Name: count, dtype: int64

--- Apetito ---
Apetito
bueno    8425
bajo     6623
NaN         1
Name: count, dtype: int64


In [14]:
categorical_missing_cols = [
    "Grupos de células de pus",
    "Bacterias",
    "Apetito"
]

for col in categorical_missing_cols:
    df_work[col] = df_work[col].fillna("desconocido")


creatinine_median = df_work["Creatinina sérica"].median()

df_work["Creatinina sérica"] = (
    df_work["Creatinina sérica"].fillna(creatinine_median)
)

print("Missing values after treatment:")
print(df_work.isna().sum()[df_work.isna().sum() > 0])

print("\nCreatinine median used:", creatinine_median)
print("\nWorking dataset shape:", df_work.shape)
print("Original dataset shape:", df_raw.shape)

Missing values after treatment:
Series([], dtype: int64)

Creatinine median used: 1.4

Working dataset shape: (15049, 25)
Original dataset shape: (15051, 21)


In [15]:
numeric_cols = [
    "Edad",
    "Presión arterial",
    "Gravedad específica de la orina",
    "Albúmina en orina",
    "Azúcar en orina",
    "Glucosa en sangre aleatoria",
    "Urea en sangre",
    "Creatinina sérica",
    "Sodio en sangre",
    "Potasio en sangre",
    "Hemoglobina",
    "Volumen corpuscular empaquetado",
    "Recuento de glóbulos blancos",
    "Recuento de glóbulos rojos"
]

outlier_results = []

for col in numeric_cols:
    s = pd.to_numeric(df_work[col], errors="coerce").dropna()

    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_results.append({
        "Column": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower Bound": lower,
        "Upper Bound": upper,
        "Outliers Count": ((s < lower) | (s > upper)).sum(),
        "Minimum": s.min(),
        "Maximum": s.max()
    })

outlier_df = pd.DataFrame(outlier_results)

display(outlier_df.round(2))

,Column,Q1,Q3,IQR,Lower Bound,Upper Bound,Outliers Count,Minimum,Maximum
0,Edad,45.00,65.00,20.00,15.00,95.00,850,2.00,90.00
1,Presión arterial,80.00,107.00,27.00,39.50,147.50,200,50.00,180.00
2,Gravedad específica de la orina,12.63,1020.00,1007.37,-1498.42,2531.06,0,10.00,1025.00
3,Albúmina en orina,0.00,452.20,452.20,-678.30,1130.50,0,0.00,989.90
4,Azúcar en orina,0.00,200.00,200.00,-300.00,500.00,1808,0.00,1000.00
5,Glucosa en sangre aleatoria,105.00,208.00,103.00,-49.50,362.50,0,70.00,300.00
6,Urea en sangre,25.00,93.50,68.50,-77.75,196.25,4,10.00,200.00
7,Creatinina sérica,0.94,3.25,2.31,-2.52,6.72,1536,0.40,76.00
8,Sodio en sangre,135.00,143.00,8.00,123.00,155.00,52,120.00,150.00
9,Potasio en sangre,4.79,6.09,1.30,2.84,8.04,0,3.50,7.50


In [16]:
cols_to_review = {
    "Presión arterial": lambda s: s > 147.5,
    "Azúcar en orina": lambda s: s > 500,
    "Urea en sangre": lambda s: s > 196.25,
    "Creatinina sérica": lambda s: s > 6.72,
    "Sodio en sangre": lambda s: s < 123
}

for col, condition in cols_to_review.items():
    values = pd.to_numeric(df_work[col], errors="coerce")
    mask = condition(values)

    print(f"\n{'=' * 50}")
    print(f"Column: {col}")
    print("Number of flagged rows:", mask.sum())
    print("\nHighest / lowest flagged values:")

    if col == "Sodio en sangre":
        print(values[mask].sort_values().value_counts().sort_index())
    else:
        print(values[mask].sort_values(ascending=False).value_counts().sort_index())


Column: Presión arterial
Number of flagged rows: 200

Highest / lowest flagged values:
Presión arterial
148     2
149     3
150    10
151     1
152     2
153     3
154     9
155     6
156     3
157     4
158    11
159     6
160     2
161     1
162     6
163     2
164     7
165     6
166     6
167     4
168     6
169     5
170     5
171     2
172     8
173     5
174     8
175     1
176     5
177     6
178     6
179     2
180    47
Name: count, dtype: int64

Column: Azúcar en orina
Number of flagged rows: 1808

Highest / lowest flagged values:
Azúcar en orina
501      4
502      4
503      1
504      3
505      6
506      3
507      7
508      4
509      4
510      5
511      3
512      5
513      2
514      7
515      8
516      8
517      2
518      8
519      1
520      2
521      2
522      2
523      3
524      1
525      3
526      4
527      6
528      5
529      3
530      1
531      4
532      1
533      2
534      1
535      5
536      5
537      3
539      3
540      2
541   

In [17]:
id_col = "Identificación del paciente"
target = "Insuficiencia renal crónica"

cols_to_review = [
    "Gravedad específica de la orina",
    "Albúmina en orina",
    "Azúcar en orina"
]

for col in cols_to_review:
    print(f"\n{'=' * 60}")
    print(f"Column: {col}")

    print("\nSmallest values:")
    display(
        df_work.nsmallest(10, col)[[id_col, col, target]]
    )

    print("\nLargest values:")
    display(
        df_work.nlargest(10, col)[[id_col, col, target]]
    )


Column: Gravedad específica de la orina

Smallest values:


,Identificación del paciente,Gravedad específica de la orina,Insuficiencia renal crónica
8624,8624,10.00,si
9355,9355,10.00,si
10105,10105,10.00,si
10363,10363,10.00,si
11219,11219,10.00,si
11423,11423,10.00,si
11764,11764,10.00,si
11815,11815,10.00,si
12686,12686,10.00,no
14286,14286,10.00,no



Largest values:


,Identificación del paciente,Gravedad específica de la orina,Insuficiencia renal crónica
247,247,1025.00,no
263,263,1025.00,no
318,318,1025.00,no
319,319,1025.00,no
336,336,1025.00,no
340,340,1025.00,no
354,354,1025.00,no
370,370,1025.00,no
371,371,1025.00,no
372,372,1025.00,no



Column: Albúmina en orina

Smallest values:


...

,Identificación del paciente,Albúmina en orina,Insuficiencia renal crónica
7982,7982,989.90,si
11740,11740,989.80,si
13491,13491,989.80,no
13973,13973,989.80,no
8930,8930,989.60,si
10847,10847,989.50,si
8119,8119,989.40,no
10183,10183,989.30,si
10477,10477,989.10,si
11540,11540,989.00,si



Column: Azúcar en orina

Smallest values:


,Identificación del paciente,Azúcar en orina,Insuficiencia renal crónica
240,240,0,no
241,241,0,no
248,248,0,no
249,249,0,no
254,254,0,no
255,255,0,no
256,256,0,no
257,257,0,no
261,261,0,no
265,265,0,no



Largest values:


,Identificación del paciente,Azúcar en orina,Insuficiencia renal crónica
9964,9964,1000,no
13999,13999,1000,no
9948,9948,999,no
12216,12216,999,no
13524,13524,999,no
14267,14267,999,no
14315,14315,999,no
12901,12901,998,no
14425,14425,998,no
14585,14585,998,no


In [18]:
numeric_cols = [
    "Edad",
    "Presión arterial",
    "Glucosa en sangre aleatoria",
    "Urea en sangre",
    "Creatinina sérica",
    "Sodio en sangre",
    "Potasio en sangre",
    "Hemoglobina",
    "Volumen corpuscular empaquetado",
    "Recuento de glóbulos blancos",
    "Recuento de glóbulos rojos"
]

for col in numeric_cols:
    s = pd.to_numeric(df_work[col], errors="coerce")

    print(f"\n--- {col} ---")
    print("Unique values:", s.nunique())
    print("Minimum:", s.min())
    print("Maximum:", s.max())
    print("Zero values:", (s == 0).sum())
    print("Negative values:", (s < 0).sum())
    print("Missing values:", s.isna().sum())


--- Edad ---
Unique values: 75
Minimum: 2
Maximum: 90
Zero values: 0
Negative values: 0
Missing values: 0

--- Presión arterial ---
Unique values: 93
Minimum: 50
Maximum: 180
Zero values: 0
Negative values: 0
Missing values: 0

--- Glucosa en sangre aleatoria ---
Unique values: 231
Minimum: 70
Maximum: 300
Zero values: 0
Negative values: 0
Missing values: 0

--- Urea en sangre ---
Unique values: 1165
Minimum: 10.0
Maximum: 200.0
Zero values: 0
Negative values: 0
Missing values: 0

--- Creatinina sérica ---
Unique values: 353
Minimum: 0.4
Maximum: 76.0
Zero values: 0
Negative values: 0
Missing values: 0

--- Sodio en sangre ---
Unique values: 256
Minimum: 120.0
Maximum: 150.0
Zero values: 0
Negative values: 0
Missing values: 0

--- Potasio en sangre ---
Unique values: 221
Minimum: 3.5
Maximum: 7.5
Zero values: 0
Negative values: 0
Missing values: 0

--- Hemoglobina ---
Unique values: 47
Minimum: 6.0
Maximum: 16.0
Zero values: 0
Negative values: 0
Missing values: 0

--- Volumen corpuscu

In [19]:
checks = {
    "Presión arterial": lambda s: s > 147.5,
    "Azúcar en orina": lambda s: s > 500,
    "Urea en sangre": lambda s: s > 196.25,
    "Creatinina sérica": lambda s: s > 6.72,
    "Sodio en sangre": lambda s: s < 123
}

for col, condition in checks.items():
    values = pd.to_numeric(df_work[col], errors="coerce")
    flagged = df_work.loc[condition(values), [id_col, col, target]]

    print(f"\n--- {col} ---")
    print("Flagged records:", len(flagged))
    print("Distinct flagged values:", flagged[col].nunique())
    print(flagged.sort_values(col).head(10))
    print("...")
    print(flagged.sort_values(col).tail(10))



--- Presión arterial ---
Flagged records: 200
Distinct flagged values: 33
     Identificación del paciente  Presión arterial Insuficiencia renal crónica
89                            89               148                          si
59                            59               148                          si
49                            49               149                          si
71                            71               149                          si
230                          230               149                          si
28                            28               150                          si
41                            41               150                          si
45                            45               150                          si
155                          155               150                          si
135                          135               150                          si
...
       Identificación del paciente  Presión arterial

In [20]:
print("Current shape:", df_work.shape)

print("\nColumn names and data types:")
print(df_work.dtypes.to_string())

print("\nHelper columns:")
helper_cols = [
    col for col in df_work.columns
    if col in ["SG_unusual", "Albumin_unusual", "Sugar_unusual",
               "ID_group", "ID_range"]
]
print(helper_cols)

print("\nTarget distribution:")
print(df_work[target].value_counts(dropna=False))

print("\nID uniqueness:")
print("Unique IDs:", df_work[id_col].nunique())
print("Total rows:", len(df_work))
print("Duplicate IDs:", df_work[id_col].duplicated().sum())


Current shape: (15049, 25)

Column names and data types:
Identificación del paciente           int64
Edad                                  int64
Presión arterial                      int64
Gravedad específica de la orina     float64
Albúmina en orina                   float64
Azúcar en orina                       int64
Glóbulos rojos                          str
Células de pus                          str
Grupos de células de pus                str
Bacterias                               str
Glucosa en sangre aleatoria           int64
Urea en sangre                      float64
Creatinina sérica                   float64
Sodio en sangre                     float64
Potasio en sangre                   float64
Hemoglobina                         float64
Volumen corpuscular empaquetado     float64
Recuento de glóbulos blancos          int64
Recuento de glóbulos rojos          float64
Apetito                                 str
Insuficiencia renal crónica             str
SG_unusual         

In [21]:
helper_cols = [
    "SG_unusual",
    "Albumin_unusual",
    "Sugar_unusual",
    "ID_group"
]

df_work = df_work.drop(
    columns=helper_cols,
    errors="ignore"
).copy()

print("Shape after removing EDA helper columns:", df_work.shape)
print("\nRemaining columns:")
print(df_work.columns.tolist())


Shape after removing EDA helper columns: (15049, 21)

Remaining columns:
['Identificación del paciente', 'Edad', 'Presión arterial', 'Gravedad específica de la orina', 'Albúmina en orina', 'Azúcar en orina', 'Glóbulos rojos', 'Células de pus', 'Grupos de células de pus', 'Bacterias', 'Glucosa en sangre aleatoria', 'Urea en sangre', 'Creatinina sérica', 'Sodio en sangre', 'Potasio en sangre', 'Hemoglobina', 'Volumen corpuscular empaquetado', 'Recuento de glóbulos blancos', 'Recuento de glóbulos rojos', 'Apetito', 'Insuficiencia renal crónica']


In [22]:
categorical_cols = [
    "Glóbulos rojos",
    "Células de pus",
    "Grupos de células de pus",
    "Bacterias",
    "Apetito",
    "Insuficiencia renal crónica"
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print("Unique values:", df_work[col].nunique(dropna=False))
    print(df_work[col].value_counts(dropna=False).to_string())


--- Glóbulos rojos ---
Unique values: 2
Glóbulos rojos
normal     7689
anormal    7360

--- Células de pus ---
Unique values: 2
Células de pus
normal     7654
anormal    7395

--- Grupos de células de pus ---
Unique values: 3
Grupos de células de pus
ausente        12595
presente        2445
desconocido        9

--- Bacterias ---
Unique values: 3
Bacterias
ausente        13731
presente        1312
desconocido        6

--- Apetito ---
Unique values: 3
Apetito
bueno          8425
bajo           6623
desconocido       1

--- Insuficiencia renal crónica ---
Unique values: 2
Insuficiencia renal crónica
no    7696
si    7353


In [23]:
categorical_cols = [
    "Glóbulos rojos",
    "Células de pus",
    "Grupos de células de pus",
    "Bacterias",
    "Apetito",
    "Insuficiencia renal crónica"
]

for col in categorical_cols:
    s = df_work[col].dropna().astype(str)

    has_extra_spaces = s.ne(s.str.strip())
    has_case_variations = s.ne(s.str.lower())

    print(f"\n--- {col} ---")
    print("Values with extra spaces:", has_extra_spaces.sum())
    print("Values differing from lowercase:", has_case_variations.sum())
    print("Unique normalized values:",
          s.str.strip().str.lower().nunique())



--- Glóbulos rojos ---
Values with extra spaces: 0
Values differing from lowercase: 0
Unique normalized values: 2

--- Células de pus ---
Values with extra spaces: 0
Values differing from lowercase: 0
Unique normalized values: 2

--- Grupos de células de pus ---
Values with extra spaces: 0
Values differing from lowercase: 0
Unique normalized values: 3

--- Bacterias ---
Values with extra spaces: 0
Values differing from lowercase: 0
Unique normalized values: 3

--- Apetito ---
Values with extra spaces: 0
Values differing from lowercase: 0
Unique normalized values: 3

--- Insuficiencia renal crónica ---
Values with extra spaces: 0
Values differing from lowercase: 0
Unique normalized values: 2


In [24]:
print("Exact duplicate rows:", df_work.duplicated().sum())

print("Missing values:", df_work.isna().sum().sum())

print("Current shape:", df_work.shape)
print("Original raw shape:", df_raw.shape)

Exact duplicate rows: 0
Missing values: 0
Current shape: (15049, 21)
Original raw shape: (15051, 21)


In [25]:
print(df_work.dtypes.to_string())

print("\nNumeric columns:")
print(df_work.select_dtypes(include="number").columns.tolist())

print("\nNon-numeric columns:")
print(df_work.select_dtypes(exclude="number").columns.tolist())

Identificación del paciente          int64
Edad                                 int64
Presión arterial                     int64
Gravedad específica de la orina    float64
Albúmina en orina                  float64
Azúcar en orina                      int64
Glóbulos rojos                         str
Células de pus                         str
Grupos de células de pus               str
Bacterias                              str
Glucosa en sangre aleatoria          int64
Urea en sangre                     float64
Creatinina sérica                  float64
Sodio en sangre                    float64
Potasio en sangre                  float64
Hemoglobina                        float64
Volumen corpuscular empaquetado    float64
Recuento de glóbulos blancos         int64
Recuento de glóbulos rojos         float64
Apetito                                str
Insuficiencia renal crónica            str

Numeric columns:
['Identificación del paciente', 'Edad', 'Presión arterial', 'Gravedad específic

In [26]:
binary_mapping = {
    "normal": 0,
    "anormal": 1
}

df_work["Glóbulos rojos"] = (
    df_work["Glóbulos rojos"].map(binary_mapping)
)

df_work["Células de pus"] = (
    df_work["Células de pus"].map(binary_mapping)
)


onehot_cols = [
    "Grupos de células de pus",
    "Bacterias",
    "Apetito"
]

df_work = pd.get_dummies(
    df_work,
    columns=onehot_cols,
    dtype=int
)


target_mapping = {
    "no": 0,
    "si": 1
}

df_work["Insuficiencia renal crónica"] = (
    df_work["Insuficiencia renal crónica"].map(target_mapping)
)


print("Encoded shape:", df_work.shape)

print("\nData types:")
print(df_work.dtypes.to_string())

print("\nMissing values after encoding:",
      df_work.isna().sum().sum())

print("\nTarget distribution:")
print(df_work["Insuficiencia renal crónica"].value_counts())

print("\nFirst 5 rows:")
display(df_work.head())


Encoded shape: (15049, 27)

Data types:
Identificación del paciente               int64
Edad                                      int64
Presión arterial                          int64
Gravedad específica de la orina         float64
Albúmina en orina                       float64
Azúcar en orina                           int64
Glóbulos rojos                            int64
Células de pus                            int64
Glucosa en sangre aleatoria               int64
Urea en sangre                          float64
Creatinina sérica                       float64
Sodio en sangre                         float64
Potasio en sangre                       float64
Hemoglobina                             float64
Volumen corpuscular empaquetado         float64
Recuento de glóbulos blancos              int64
Recuento de glóbulos rojos              float64
Insuficiencia renal crónica               int64
Grupos de células de pus_ausente          int64
Grupos de células de pus_desconocido      int64


,Identificación del paciente,Edad,Presión arterial,Gravedad específica de la orina,Albúmina en orina,Azúcar en orina,Glóbulos rojos,Células de pus,Glucosa en sangre aleatoria,Urea en sangre,Creatinina sérica,Sodio en sangre,Potasio en sangre,Hemoglobina,Volumen corpuscular empaquetado,Recuento de glóbulos blancos,Recuento de glóbulos rojos,Insuficiencia renal crónica,Grupos de células de pus_ausente,Grupos de células de pus_desconocido,Grupos de células de pus_presente,Bacterias_ausente,Bacterias_desconocido,Bacterias_presente,Apetito_bajo,Apetito_bueno,Apetito_desconocido
0,0,48,178,1010.00,2.00,2,1,1,298,195.00,9.08,124.00,6.00,8.00,25.00,11319,3.80,1,0,0,1,0,0,1,1,0,0
1,1,7,138,1014.00,1.00,4,1,1,187,111.00,7.88,133.00,6.80,11.00,27.00,11018,2.90,1,0,0,1,0,0,1,1,0,0
2,2,62,165,1014.00,2.00,5,1,1,286,149.00,12.89,130.00,6.00,8.00,27.00,5479,4.40,1,0,0,1,0,0,1,1,0,0
3,3,48,141,1007.00,4.00,4,1,1,165,98.00,5.02,133.00,5.80,11.00,35.00,7817,3.20,1,0,0,1,0,0,1,1,0,0
4,4,51,175,1015.00,4.00,5,1,1,179,152.00,7.32,132.00,6.80,9.00,30.00,3055,3.90,1,0,0,1,0,0,1,1,0,0


In [27]:
df_work = df_work.drop(columns=["Identificación del paciente"])

print(df_work.shape)

(15049, 26)


In [28]:
print("Dataset Shape:", df_work.shape)


print("\nTotal Missing Values:")
print(df_work.isna().sum().sum())


print("\nExact Duplicate Rows:")
print(df_work.duplicated().sum())


print("\nData Types:")
print(df_work.dtypes)


target = "Insuficiencia renal crónica"

print("\nTarget Distribution:")
print(df_work[target].value_counts(dropna=False))


print("\nUnique Target Values:")
print(df_work[target].unique())


id_col = "Identificación del paciente"

print("\nID Column Exists:", id_col in df_work.columns)


print("\nFinal Columns:")
print(df_work.columns.tolist())

Dataset Shape: (15049, 26)

Total Missing Values:
0

Exact Duplicate Rows:
1

Data Types:
Edad                                      int64
Presión arterial                          int64
Gravedad específica de la orina         float64
Albúmina en orina                       float64
Azúcar en orina                           int64
Glóbulos rojos                            int64
Células de pus                            int64
Glucosa en sangre aleatoria               int64
Urea en sangre                          float64
Creatinina sérica                       float64
Sodio en sangre                         float64
Potasio en sangre                       float64
Hemoglobina                             float64
Volumen corpuscular empaquetado         float64
Recuento de glóbulos blancos              int64
Recuento de glóbulos rojos              float64
Insuficiencia renal crónica               int64
Grupos de células de pus_ausente          int64
Grupos de células de pus_desconocido      int6

In [29]:
duplicate_mask = df_work.duplicated(keep=False)

duplicate_indices = df_work.index[duplicate_mask]

print("Number of rows involved in duplication:")
print(len(duplicate_indices))

print("\nDuplicated rows in df_work:")
display(df_work.loc[duplicate_indices])

id_col = "Identificación del paciente"

print("\nOriginal IDs and target values:")
display(df_raw.loc[duplicate_indices, [id_col, "Insuficiencia renal crónica"]])

Number of rows involved in duplication:
2

Duplicated rows in df_work:


,Edad,Presión arterial,Gravedad específica de la orina,Albúmina en orina,Azúcar en orina,Glóbulos rojos,Células de pus,Glucosa en sangre aleatoria,Urea en sangre,Creatinina sérica,Sodio en sangre,Potasio en sangre,Hemoglobina,Volumen corpuscular empaquetado,Recuento de glóbulos blancos,Recuento de glóbulos rojos,Insuficiencia renal crónica,Grupos de células de pus_ausente,Grupos de células de pus_desconocido,Grupos de células de pus_presente,Bacterias_ausente,Bacterias_desconocido,Bacterias_presente,Apetito_bajo,Apetito_bueno,Apetito_desconocido
1531,46,109,1017.00,1.00,0,0,0,107,27.00,1.07,139.00,3.90,16.00,47.00,5379,4.70,0,1,0,0,1,0,0,0,1,0
15040,46,109,1017.00,1.00,0,0,0,107,27.00,1.07,139.00,3.90,16.00,47.00,5379,4.70,0,1,0,0,1,0,0,0,1,0



Original IDs and target values:


,Identificación del paciente,Insuficiencia renal crónica
1531,1531,no
15040,15040,no


In [30]:
col = "Presión arterial"

Q1 = df_work[col].quantile(0.25)
Q3 = df_work[col].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_work[
    (df_work[col] < lower_bound) |
    (df_work[col] > upper_bound)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

print("\nOutlier Count:", len(outliers))

print("\nOutlier Values Frequency:")
print(outliers[col].value_counts().sort_index())

print("\nOutlier Summary:")
print(outliers[col].describe())

Q1: 80.0
Q3: 107.0
IQR: 27.0
Lower Bound: 39.5
Upper Bound: 147.5

Outlier Count: 200

Outlier Values Frequency:
Presión arterial
148     2
149     3
150    10
151     1
152     2
153     3
154     9
155     6
156     3
157     4
158    11
159     6
160     2
161     1
162     6
163     2
164     7
165     6
166     6
167     4
168     6
169     5
170     5
171     2
172     8
173     5
174     8
175     1
176     5
177     6
178     6
179     2
180    47
Name: count, dtype: int64

Outlier Summary:
count   200.00
mean    167.62
std      10.40
min     148.00
25%     158.00
50%     168.50
75%     178.00
max     180.00
Name: Presión arterial, dtype: float64


In [31]:
col = "Azúcar en orina"

Q1 = df_work[col].quantile(0.25)
Q3 = df_work[col].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_work[
    (df_work[col] < lower_bound) |
    (df_work[col] > upper_bound)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

print("\nOutlier Count:", len(outliers))

print("\nOutlier Values Frequency:")
print(outliers[col].value_counts().sort_index())

print("\nOutlier Summary:")
print(outliers[col].describe())

Q1: 0.0
Q3: 200.0
IQR: 200.0
Lower Bound: -300.0
Upper Bound: 500.0

Outlier Count: 1808

Outlier Values Frequency:
Azúcar en orina
501      4
502      4
503      1
504      3
505      6
506      3
507      7
508      4
509      4
510      5
511      3
512      5
513      2
514      7
515      8
516      8
517      2
518      8
519      1
520      2
521      2
522      2
523      3
524      1
525      3
526      4
527      6
528      5
529      3
530      1
531      4
532      1
533      2
534      1
535      5
536      5
537      3
539      3
540      2
541      2
542      3
543      4
544      3
545      5
546      3
547      7
548      2
549      4
550      6
551      2
552      5
553      6
554      2
555      3
556      1
557      6
558      3
559      4
560      4
561      1
562      7
563      6
564      5
565      6
566      2
567      5
568      5
569      2
570      7
571      5
572      6
573      3
574      2
575      3
576      3
577      4
578      2
579      6
581      4

In [32]:
col = "Azúcar en orina"
target = "Insuficiencia renal crónica"


bins = [-1, 5, 100, 200, 300, 400, 500, 750, 1000]
labels = [
    "0–5", "6–100", "101–200", "201–300",
    "301–400", "401–500", "501–750", "751–1000"
]

df_work["Sugar_Range"] = pd.cut(
    df_work[col],
    bins=bins,
    labels=labels
)

print("Count per range:")
print(df_work["Sugar_Range"].value_counts().sort_index())

print("\nTarget distribution per range:")
print(pd.crosstab(
    df_work["Sugar_Range"],
    df_work[target],
    margins=True
))

print("\nData type:", df_work[col].dtype)
print("\nMinimum:", df_work[col].min())
print("Maximum:", df_work[col].max())
print("\nSample values:")
print(df_work[col].sample(20, random_state=42).sort_values().tolist())

Count per range:
Sugar_Range
0–5         7903
6–100       1400
101–200     1994
201–300     1239
301–400      360
401–500      345
501–750      907
751–1000     901
Name: count, dtype: int64

Target distribution per range:
Insuficiencia renal crónica     0     1    All
Sugar_Range                                   
0–5                          4261  3642   7903
6–100                         176  1224   1400
101–200                       365  1629   1994
201–300                       381   858   1239
301–400                       360     0    360
401–500                       345     0    345
501–750                       907     0    907
751–1000                      901     0    901
All                          7696  7353  15049

Data type: int64

Minimum: 0
Maximum: 1000

Sample values:
[0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 64, 65, 93, 109, 170, 174, 201, 272]


In [33]:
cols = [
    "Gravedad específica de la orina",
    "Albúmina en orina",
    "Azúcar en orina"
]

for col in cols:
    print(f"\n{'=' * 50}")
    print("Column:", col)
    print("Data type:", df_work[col].dtype)
    print("Unique values:", df_work[col].nunique())
    print("Minimum:", df_work[col].min())
    print("Maximum:", df_work[col].max())
    print("\nQuantiles:")
    print(df_work[col].quantile([0, 0.25, 0.5, 0.75, 1]))
    print("\nSample values:")
    print(sorted(df_work[col].dropna().sample(
        min(15, df_work[col].notna().sum()),
        random_state=42
    ).tolist()))


Column: Gravedad específica de la orina
Data type: float64
Unique values: 522
Minimum: 10.0
Maximum: 1025.0

Quantiles:
0.00     10.00
0.25     12.63
0.50   1015.00
0.75   1020.00
1.00   1025.00
Name: Gravedad específica de la orina, dtype: float64

Sample values:
[10.18, 11.31, 12.92, 14.72, 14.74, 1017.0, 1017.0, 1017.0, 1018.0, 1019.0, 1020.0, 1022.0, 1022.0, 1022.0, 1025.0]

Column: Albúmina en orina
Data type: float64
Unique values: 5050
Minimum: 0.0
Maximum: 989.9

Quantiles:
0.00     0.00
0.25     0.00
0.50     1.00
0.75   452.20
1.00   989.90
Name: Albúmina en orina, dtype: float64

Sample values:
[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 299.3, 307.3, 352.6, 415.4, 743.6]

Column: Azúcar en orina
Data type: int64
Unique values: 959
Minimum: 0
Maximum: 1000

Quantiles:
0.00      0.00
0.25      0.00
0.50      1.00
0.75    200.00
1.00   1000.00
Name: Azúcar en orina, dtype: float64

Sample values:
[0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 64, 65, 109, 174, 272]


In [35]:
col = "Creatinina sérica"
target = "Insuficiencia renal crónica"

Q1 = df_work[col].quantile(0.25)
Q3 = df_work[col].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_work[
    (df_work[col] < lower_bound) |
    (df_work[col] > upper_bound)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)
print("Outlier Count:", len(outliers))

print("\nOutlier values frequency:")
print(outliers[col].value_counts().sort_index())

print("\nTarget distribution among outliers:")
print(outliers[target].value_counts())

print("\nHighest values:")
print(
    df_work.nlargest(20, col)[[col, target]]
)

Q1: 0.94
Q3: 3.25
IQR: 2.31
Lower Bound: -2.525
Upper Bound: 6.715
Outlier Count: 1536

Outlier values frequency:
Creatinina sérica
6.72       1
6.78       1
6.80      17
6.82       1
6.95       1
7.04       1
7.10      60
7.15       1
7.20      67
7.28       1
7.30     132
7.32       1
7.37       1
7.48       1
7.49       1
7.50      15
7.52       1
7.56       1
7.60       1
7.70      35
7.72       1
7.78       1
7.80       1
7.83       2
7.88       3
7.89       1
7.90       1
8.03       1
8.10       1
8.12       1
8.13       1
8.17       3
8.38       1
8.41       1
8.44       2
8.48       1
8.55       1
8.72       1
8.79       1
8.80       1
8.81       1
8.86       1
8.87       1
8.89       1
8.90       1
8.93       1
9.08       1
9.20      43
9.30      12
9.31       1
9.36       1
9.45       1
9.47       1
9.60      42
9.66       1
9.69       1
9.70      70
9.72       1
9.82       1
9.99       2
10.00     34
10.16      1
10.20     80
10.21      1
10.24      1
10.28      2
10.40     

In [37]:
col = "Sodio en sangre"
target = "Insuficiencia renal crónica"

Q1 = df_work[col].quantile(0.25)
Q3 = df_work[col].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_work[
    (df_work[col] < lower_bound) |
    (df_work[col] > upper_bound)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)
print("Outlier Count:", len(outliers))

print("\nOutlier Values Frequency:")
print(outliers[col].value_counts().sort_index())

print("\nTarget Distribution:")
print(outliers[target].value_counts())

print("\nOutlier Summary:")
print(outliers[col].describe())

Q1: 135.0
Q3: 143.0
IQR: 8.0
Lower Bound: 123.0
Upper Bound: 155.0
Outlier Count: 52

Outlier Values Frequency:
Sodio en sangre
120.00    18
121.00    17
122.00    17
Name: count, dtype: int64

Target Distribution:
Insuficiencia renal crónica
1    48
0     4
Name: count, dtype: int64

Outlier Summary:
count    52.00
mean    120.98
std       0.83
min     120.00
25%     120.00
50%     121.00
75%     122.00
max     122.00
Name: Sodio en sangre, dtype: float64


In [39]:
df_work["Age_Group"] = pd.cut(
    df_work["Edad"],
    bins=[-1, 17, 39, 59, float("inf")],
    labels=["Under_18", "18_39", "40_59", "60_plus"]
)

print(df_work["Age_Group"].value_counts().sort_index())
print("Missing:", df_work["Age_Group"].isna().sum())

Age_Group
Under_18    1092
18_39       1812
40_59       6030
60_plus     6115
Name: count, dtype: int64
Missing: 0


In [40]:
df_work.drop(columns=["Sugar_Range"], errors="ignore", inplace=True)

print("Sugar_Range exists:", "Sugar_Range" in df_work.columns)
print("Current shape:", df_work.shape)

Sugar_Range exists: False
Current shape: (15049, 27)


In [42]:
print("Final shape:", df_work.shape)

print("Missing values:", df_work.isna().sum().sum())

print("Target values:")
print(df_work["Insuficiencia renal crónica"].value_counts())

print("ID removed:", "Identificación del paciente" not in df_work.columns)

print("Temporary column removed:", "Sugar_Range" not in df_work.columns)

print("Age_Group exists:", "Age_Group" in df_work.columns)

print("Raw dataset unchanged:", df_raw.shape == (15051, 21))

Final shape: (15049, 27)
Missing values: 0
Target values:
Insuficiencia renal crónica
0    7696
1    7353
Name: count, dtype: int64
ID removed: True
Temporary column removed: True
Age_Group exists: True
Raw dataset unchanged: True


In [41]:
output_file = "kidney_cleaned.csv"

df_work.to_csv(output_file, index=False, encoding="utf-8-sig")

print("Cleaned dataset saved:", output_file)
print("Final shape:", df_work.shape)
print("Missing values:", df_work.isna().sum().sum())
print("Exact duplicate rows:", df_work.duplicated().sum())


Cleaned dataset saved: kidney_cleaned.csv
Final shape: (15049, 27)
Missing values: 0
Exact duplicate rows: 1
